# 02. Distributed Fraud Detection Model Training (Random Forest)
**Cymbal Financial Fraud Detection Pipeline**

This notebook trains a distributed `RandomForestClassifier` on the gold-layer `enriched_transactions` table to classify fraudulent events based on transaction features and dimensional risk metrics.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("Cymbal-Fraud-Model-Training") \
    .getOrCreate()

project_id = os.getenv("PROJECT_ID", "cymbal-fraud-detection")
dataset_name = os.getenv("DATASET_NAME", "transactions_dataset_evals")
models_bucket = os.getenv("MODELS_BUCKET", f"{project_id}-models")
model_export_path = f"gs://{models_bucket}/fraud_model"

print(f"Project ID: {project_id}")
print(f"Model export destination: {model_export_path}")

In [ ]:
# Load the enriched transactions dataset
try:
    # Load from BigQuery
    df = spark.read.format("bigquery") \
        .option("table", f"{project_id}.{dataset_name}.enriched_transactions") \
        .load()
    print("Loaded enriched_transactions from BigQuery.")
except Exception as e:
    print(f"Reading from local fallback table: {e}")
    import pandas as pd
    import duckdb
    # Fallback to local DuckDB/Parquet
    con = duckdb.connect("data/cymbal_fraud.duckdb")
    pdf = con.execute("SELECT * FROM enriched_transactions").df()
    df = spark.createDataFrame(pdf)

# Filter only historically labeled records for model training
train_source_df = df.filter(col("is_fraud").isNotNull()).cache()
print(f"Total labeled training records: {train_source_df.count():,}")
train_source_df.groupBy("is_fraud").count().show()

In [ ]:
# Define Categorical and Numerical features
categorical_cols = ["payment_method", "currency", "payor_country", "payee_country", "payee_category"]
numerical_cols = ["amount", "payor_risk_score", "payee_risk_score", "merchant_mcc"]

# 1. String Indexers for Categorical Columns
indexers = [
    StringIndexer(inputCol=col_name, outputCol=f"{col_name}_idx", handleInvalid="keep")
    for col_name in categorical_cols
]

# 2. One-Hot Encoders
encoders = [
    OneHotEncoder(inputCol=f"{col_name}_idx", outputCol=f"{col_name}_vec")
    for col_name in categorical_cols
]

# 3. Vector Assembler
assembler_inputs = [f"{col_name}_vec" for col_name in categorical_cols] + numerical_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="unscaled_features", handleInvalid="keep")

# 4. Feature Scaler
scaler = StandardScaler(inputCol="unscaled_features", outputCol="features", withStd=True, withMean=False)

# 5. Distributed Random Forest Classifier
rf = RandomForestClassifier(
    labelCol="is_fraud",
    featuresCol="features",
    numTrees=50,
    maxDepth=8,
    seed=42
)

# Construct ML Pipeline
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, rf])
print("ML Pipeline successfully defined with stages:", [type(s).__name__ for s in pipeline.getStages()])

In [ ]:
# Split dataset into 80% Train and 20% Test splits
train_data, test_data = train_source_df.randomSplit([0.8, 0.2], seed=42)
print(f"Training set: {train_data.count():,} rows | Test set: {test_data.count():,} rows")

# Train the model pipeline
print("Fitting RandomForest ML Pipeline...")
model = pipeline.fit(train_data)
print("Model training complete.")

In [ ]:
# Evaluate on Test Data
predictions = model.transform(test_data)

evaluator_auc = BinaryClassificationEvaluator(labelCol="is_fraud", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_pr = BinaryClassificationEvaluator(labelCol="is_fraud", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="is_fraud", predictionCol="prediction", metricName="accuracy")

auc_score = evaluator_auc.evaluate(predictions)
pr_score = evaluator_pr.evaluate(predictions)
acc_score = evaluator_acc.evaluate(predictions)

print("=== Model Evaluation Results ===")
print(f"Area Under ROC (AUC) : {auc_score:.4f}")
print(f"Area Under PR (AUPRC): {pr_score:.4f}")
print(f"Classification Acc   : {acc_score:.4f}")

In [ ]:
# Save the trained pipeline model to Cloud Storage / local path
try:
    model.write().overwrite().save(model_export_path)
    print(f"Model exported successfully to {model_export_path}")
except Exception as e:
    print(f"Cloud export bypassed: {e}")
    local_model_path = "models/fraud_model"
    model.write().overwrite().save(local_model_path)
    print(f"Model saved locally to {local_model_path}")